<a href="https://colab.research.google.com/github/mitrasujoy/ollama/blob/main/Running_Ollama_on_Google_Colab_Through_Pinggy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Consolidated Installation
This cell handles the entire environment setup by:
* Updating system packages and installing `pciutils` (for GPU inspection) and `zstd` (for decompression).
* Installing the **Ollama** framework.
* Installing Python dependencies including `pinggy` for tunneling and `open-webui` for the graphical interface.

In [ ]:
!sudo apt-get update && sudo apt-get install -y pciutils zstd
!curl -fsSL https://ollama.ai/install.sh | sh
!pip install pinggy open-webui

# Step 2: Initialize Ollama Server with Health Check
We launch the Ollama server in the background and include an automated health check. The script will wait and verify that the server is responding on port 11434 before allowing the notebook to proceed.

In [ ]:
import subprocess
import os
import time
import requests

def start_ollama_server():
    os.environ['OLLAMA_HOST'] = '0.0.0.0'
    os.environ['OLLAMA_ORIGINS'] = '*'

    # Launch server
    subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=os.environ)

    # Optimization: Wait for server to be ready before proceeding
    print("⌛ Waiting for Ollama server to initialize...")
    for i in range(10):
        try:
            requests.get("http://localhost:11434/api/tags")
            print("🚀 Ollama server is ready!")
            return
        except:
            time.sleep(2)
    print("❌ Ollama server failed to start.")

start_ollama_server()

# Step 3: API Tunnel Management
We use `pinggy` to establish a secure tunnel for the Ollama backend (Port 11434). This allows for direct API requests to the model from external services. The WebUI tunnel will be initialized later, only once the server is fully ready.

In [ ]:
import pinggy
# Only initialize the API tunnel for now
api_tunnel = pinggy.start_tunnel(forwardto="localhost:11434")
print(f"✅ API Tunnel: {api_tunnel.urls[0]}")

# Step 4: Model Preparation
This cell pulls the `llama3.2:1b` model, ensuring the required weights are downloaded to the local runtime for inference.

In [ ]:
!ollama pull llama3.2:1b


# Step 5: Verify Model Functionality
Before launching the full interface, we'll send a quick local request to the model to ensure it is loaded and responding correctly.

In [ ]:
import requests
import json

# Test the model locally
test_response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:1b",
        "prompt": "Why is the sky blue? Respond in one sentence.",
        "stream": False
    }
)

if test_response.status_code == 200:
    print("✅ Model check successful!")
    print(f"Response: {test_response.json().get('response')}")
else:
    print(f"❌ Model check failed with status: {test_response.status_code}")

# Step 6: Launch Open WebUI and Initialize Tunnel
In this final step, we:
1. Launch the Open WebUI server in a **background thread** to keep the notebook interactive.
2. Run a **looping health check** to detect when the WebUI is officially listening on port 8000.
3. **Initialize the WebUI tunnel** only after the server is ready, providing a reliable link for the graphical interface.

In [ ]:
import threading
import time
import requests
import pinggy

# Start Open WebUI in a background thread so we can continue execution
def run_webui():
    !open-webui serve --port 8000

webui_thread = threading.Thread(target=run_webui)
webui_thread.start()

print("⌛ Waiting for Open WebUI to start on port 8000...")
for i in range(30):
    try:
        requests.get("http://localhost:8000/health")
        print("🚀 Open WebUI is ready!")
        break
    except:
        time.sleep(5)
else:
    print("⚠️ WebUI is taking longer than expected to start.")

# Initialize the WebUI tunnel AFTER it is ready
webui_tunnel = pinggy.start_tunnel(forwardto="localhost:8000")
print(f"\n🔗 Access Open WebUI here: {webui_tunnel.urls[0]}")